In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as plt_sns
from scipy import stats

from models import identify_dispersion_regimes


d:\Users\Tiphaine\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\Users\Tiphaine\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
market_returns = pd.read_csv('../data/processed/benchmark_returns.csv', index_col='Date', parse_dates=True).squeeze()
rf_daily = pd.read_csv('../data/processed/rf_daily.csv', index_col='Date', parse_dates=True).squeeze()
dispersion_df = pd.read_csv('../data/processed/sector_dispersion_factors.csv', index_col='Date', parse_dates=True)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/benchmark_returns.csv'

In [ ]:
disp_signals=dispersion_df['Rolling_CSSD_21']

df=pd.concat([market_returns,rf_daily,disp_signals],axis=1,join='inner')
df.columns=['Market_Returns','RFR_Daily','Dispersion_Signal']

In [ ]:
df['Regime_T']=identify_dispersion_regimes(df['Dispersion_Signal'], threshold=1.0, window=252)
#Le rendement du jour T doit etre associé au régime connu à la fin du jour T-1
df['Regime']=df['Regime_T'].shift(1)
df=df.dropna(subset=['Regime']) #on veut garder uniquement les jours pour lesquels on connait le régime du jour précédent

print("Dispersion des régimes (jours):")
print(df['Regime'].value_counts())

In [ ]:
def stats(group):
    mean = group['Market_Returns'].mean()
    std = group['Market_Returns'].std()
    skew = group['Market_Returns'].skew()
    kurt = group['Market_Returns'].kurtosis()
    sharpe_ratio = (mean - group['RFR_Daily'].mean()) / std if std != 0 else np.nan
    return pd.Series({'Mean': mean, 'Std': std, 'Skewness': skew, 'Kurtosis': kurt, 'Sharpe_Ratio': sharpe_ratio},'Days':int(len(group)))
regime_stats = df.groupby('Regime').apply(stats).round(2)
display(regime_stats)

In [ ]:
# Cellule 5 : Visualisation impact sur la distribution (Le graphique pour le jury)
plt.figure(figsize=(10, 6))

# On filtre les valeurs extrêmes pour un bel affichage
plot_data = df[(df['Market_Return'] > -0.05) & (df['Market_Return'] < 0.05)]

plt_sns.kdeplot(data=plot_data, x='Market_Return', hue='Regime', 
                common_norm=False, fill=True, alpha=0.3,
                palette={'High': 'red', 'Normal': 'gray', 'Low': 'blue'})

plt.title("Distribution des rendements journaliers du S&P 500 selon le Régime de Dispersion", fontsize=14)
plt.xlabel("Rendement journalier")
plt.ylabel("Densité")
plt.axvline(0, color='black', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
#Est-ce que la différence de volatilité est statistiquement significative entre les régimes ? On peut faire un test de Levene pour comparer les variances des rendements entre les régimes.


levene_test = stats.levene(df[df['Regime'] == 'High']['Market_Returns'],
                           df[df['Regime'] == 'Normal']['Market_Returns'],
                           df[df['Regime'] == 'Low']['Market_Returns'])
print("Test de Levene:")
print(f"Statistique: {levene_test.statistic}")
print(f"p-value: {levene_test.pvalue}")